In [ ]:
!pip install -q "executorch==1.1.0" huggingface_hub pytorch-tokenizers

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
!pip list 2>/dev/null | grep -i -E "executorch|torch"

executorch                            1.1.0
pytorch_tokenizers                    1.6.0
torch                                 2.11.0+cpu
torchao                               0.15.0
torchaudio                            2.11.0+cpu
torchcodec                            0.11.0+cpu
torchdata                             0.11.0
torchsummary                          1.5.1
torchtune                             0.6.1
torchvision                           0.26.0+cpu


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from huggingface_hub import login, snapshot_download
login()
snapshot_download("meta-llama/Llama-3.2-1B-Instruct", allow_patterns=["original/*"],
                  local_dir="/content/waste-assistant/llama-1b")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

'/content/waste-assistant/llama-1b'

In [ ]:
%%writefile /content/waste-assistant/export_config.yaml
base:
  model_class: llama3_2
  checkpoint: /content/waste-assistant/llama-1b/original/consolidated.00.pth
  params: /content/waste-assistant/llama-1b/original/params.json
  metadata: '{"get_bos_id":128000, "get_eos_ids":[128009, 128001]}'
model:
  use_kv_cache: True
  use_sdpa_with_kv_cache: True
quantization:
  embedding_quantize: 4,32
  qmode: 8da4w
backend:
  xnnpack:
    enabled: True
    extended_ops: True

Writing /content/waste-assistant/export_config.yaml


In [ ]:
%%bash
cd /content/waste-assistant
python -m executorch.extension.llm.export.export_llm --config export_config.yaml \
    +export.output_name="llama3_2_1b_v110.pte" \
    +export.max_seq_length=1024 +export.max_context_length=1024
ls -l *.pte

Skipping import of cpp extensions due to incompatible torch version 2.11.0+cpu for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
I tokenizers:regex.cpp:40] Registering override fallback regex
Traceback (most recent call last):
  File "<frozen runpy>", line 194, in _run_module_as_main
  File "<frozen runpy>", line 114, in _get_module_details
  File "/usr/local/lib/python3.13/dist-packages/executorch/extension/llm/export/__init__.py", line 7, in <module>
    from .builder import LLMEdgeManager
  File "/usr/local/lib/python3.13/dist-packages/executorch/extension/llm/export/builder.py", line 39, in <module>
    from torchao.quantization.pt2e.quantize_pt2e import convert_pt2e, prepare_pt2e
  File "/usr/local/lib/python3.13/dist-packages/torchao/quantization/pt2e/quantize_pt2e.py", line 44, in <module>
    quantizer: Union[Quantizer, torch.ao.quantization.quantizer.quantizer.Quantizer],
                                ^^^^^^^^^^^^^^^^^^

CalledProcessError: Command 'b'cd /content/waste-assistant\npython -m executorch.extension.llm.export.export_llm --config export_config.yaml \\\n    +export.output_name="llama3_2_1b_v110.pte" \\\n    +export.max_seq_length=1024 +export.max_context_length=1024\nls -l *.pte\n'' returned non-zero exit status 2.

In [ ]:
!pip install -q "torch==2.9.1" "executorch==1.1.0" --extra-index-url https://download.pytorch.org/whl/cpu

ERROR: Cannot install executorch==1.1.0, executorch==1.1.0+cpu and torch==2.9.1 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [ ]:
!pip download "executorch==1.1.0" --no-deps -d /tmp/et -q
!unzip -p /tmp/et/*.whl '*/METADATA' | grep -i -E "requires-dist: (torch|torchao)"


Requires-Dist: torchao==0.15.0
Requires-Dist: torch>=2.10.0


In [ ]:
!pip install -q "torch==2.10.0" "torchao==0.15.0" "executorch==1.1.0" --extra-index-url https://download.pytorch.org/whl/cpu


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.26.0+cpu requires torch==2.11.0, but you have torch 2.10.0+cpu which is incompatible.


In [ ]:
!pip list 2>/dev/null | grep -i -E "^(executorch|torch|torchao) "

executorch                            1.1.0
torch                                 2.10.0+cpu
torchao                               0.15.0


In [ ]:
import torch.ao.quantization.quantizer
from executorch.extension.llm.export.builder import LLMEdgeManager
print("imports ok")

imports ok


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip list 2>/dev/null | grep -i -E "^(executorch|torch|torchao) "

executorch                            1.1.0
torch                                 2.10.0+cpu
torchao                               0.15.0


In [ ]:
%%bash
ls /content/waste-assistant /content/waste-assistant/llama-1b/original

/content/waste-assistant:
export_config.yaml
llama-1b

/content/waste-assistant/llama-1b/original:
consolidated.00.pth
params.json
tokenizer.model


In [ ]:
%%bash
cd /content/waste-assistant
python -m executorch.extension.llm.export.export_llm --config export_config.yaml \
    +export.output_name="llama3_2_1b_v110.pte" \
    +export.max_seq_length=1024 +export.max_context_length=1024
ls -l *.pte

[2026-11-04 12:09:23,736][root][INFO] - Applying quantizers: []
[2026-11-04 12:09:39,942][root][INFO] - Checkpoint dtype: torch.bfloat16
quantize ('tok_embeddings', Embedding(128256, 2048)) with group_size 32, bitwidth 4
[2026-11-04 12:17:26,839][root][INFO] - Replacing KVCache with CustomKVCache. This modifies the model in place.
[2026-11-04 12:17:26,961][root][INFO] - Looking for libcustom_ops_aot_lib.so in /usr/local/lib/python3.13/dist-packages/executorch/extension/llm/custom_ops
[2026-11-04 12:17:26,967][root][INFO] - Loading custom ops library: /usr/local/lib/python3.13/dist-packages/executorch/extension/llm/custom_ops/libcustom_ops_aot_lib.so
[2026-11-04 12:17:27,025][root][INFO] - Model after source transforms: Transformer(
  (tok_embeddings): QuantizedGroupEmbedding()
  (layers): ModuleList(
    (0-15): 16 x TransformerBlock(
      (attention): AttentionMHA(
        (wq): Linear(in_features=2048, out_features=2048, weight=IntxUnpackedToInt8Tensor(target_dtype=torch.int4, block

Skipping import of cpp extensions due to incompatible torch version 2.10.0+cpu for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
I tokenizers:regex.cpp:40] Registering override fallback regex
/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_le

In [ ]:
%%bash
ls -l /content/waste-assistant/*.pte
cp /content/waste-assistant/llama3_2_1b_v110.pte "/content/drive/MyDrive/Waste assistant/"

-rw-r--r-- 1 root root 802530048 Nov  4 12:29 /content/waste-assistant/llama3_2_1b_v110.pte


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
!rm -rf /content/data
!mkdir -p /content/data
!unzip -q "/content/drive/MyDrive/Waste assistant/raw.zip" -d /content/data

# find the folder that holds the class folders
ROOT = None
for dp, dirs, _ in os.walk('/content/data'):
    if 'ewaste' in dirs:
        ROOT = dp
        break
print("ROOT:", ROOT)

EXT = {'.jpg','.jpeg','.png','.bmp','.webp'}
for c in sorted(os.listdir(ROOT)):
    n = sum(1 for dp,_,fs in os.walk(f'{ROOT}/{c}') for f in fs if os.path.splitext(f)[1].lower() in EXT)
    print(f'{c:28s} {n}')

ROOT: /content/data/RAW
ewaste                       293
glass                        300
hazardous                    197
metal                        300
organic                      164
paper_cardboard              297
plastic_bottle_container     205
plastic_film                 26


In [ ]:
import random, shutil
from PIL import Image
random.seed(42)
OUT = '/content/ds'
shutil.rmtree(OUT, ignore_errors=True)
classes = sorted(d for d in os.listdir(ROOT) if os.path.isdir(f'{ROOT}/{d}'))

for c in classes:
    files = [os.path.join(dp, f) for dp,_,fs in os.walk(f'{ROOT}/{c}')
             for f in fs if os.path.splitext(f)[1].lower() in EXT]
    good = []
    for p in files:
        try:
            with Image.open(p) as im:
                im.verify()
            good.append(p)
        except Exception:
            pass
    random.shuffle(good)
    n = len(good); a = int(.70*n); b = int(.85*n)
    parts = {'train': good[:a], 'val': good[a:b], 'test': good[b:]}
    for k, v in parts.items():
        os.makedirs(f'{OUT}/{k}/{c}', exist_ok=True)
        for i, p in enumerate(v):
            shutil.copy(p, f'{OUT}/{k}/{c}/{i:05d}{os.path.splitext(p)[1].lower()}')
    print(c, {k: len(v) for k, v in parts.items()}, 'skipped:', len(files)-len(good))

ewaste {'train': 205, 'val': 44, 'test': 44} skipped: 0
glass {'train': 210, 'val': 45, 'test': 45} skipped: 0
hazardous {'train': 137, 'val': 30, 'test': 30} skipped: 0
metal {'train': 210, 'val': 45, 'test': 45} skipped: 0
organic {'train': 114, 'val': 25, 'test': 25} skipped: 0
paper_cardboard {'train': 207, 'val': 45, 'test': 45} skipped: 0
plastic_bottle_container {'train': 143, 'val': 31, 'test': 31} skipped: 0
plastic_film {'train': 18, 'val': 4, 'test': 4} skipped: 0


In [ ]:
import torch, torch.nn as nn, numpy as np, json
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev)

mean, std = [0.485,0.456,0.406], [0.229,0.224,0.225]
tr = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.3,0.3,0.3),
    transforms.ToTensor(), transforms.Normalize(mean,std)])
ev = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(), transforms.Normalize(mean,std)])

train_ds = datasets.ImageFolder(f'{OUT}/train', tr)
val_ds   = datasets.ImageFolder(f'{OUT}/val', ev)
test_ds  = datasets.ImageFolder(f'{OUT}/test', ev)
K = len(train_ds.classes)
print(train_ds.classes)

train_dl = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_dl   = torch.utils.data.DataLoader(val_ds, batch_size=64, num_workers=2)
test_dl  = torch.utils.data.DataLoader(test_ds, batch_size=64, num_workers=2)

# class weights so small classes count more
cnt = np.bincount(train_ds.targets, minlength=K)
w = torch.tensor(len(train_ds)/(K*cnt), dtype=torch.float).to(dev)

model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, K)
model.to(dev)

EPOCHS = 12
opt = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
loss_fn = nn.CrossEntropyLoss(weight=w)

def evaluate(dl):
    model.eval(); P=[]; L=[]
    with torch.no_grad():
        for x,y in dl:
            P += model(x.to(dev)).argmax(1).cpu().tolist(); L += y.tolist()
    return np.mean(np.array(P)==np.array(L)), P, L

best, best_state = 0, None
for ep in range(EPOCHS):
    model.train()
    for x,y in train_dl:
        x,y = x.to(dev), y.to(dev)
        opt.zero_grad(); loss_fn(model(x), y).backward(); opt.step()
    sched.step()
    acc,_,_ = evaluate(val_dl)
    print(f'epoch {ep+1}/{EPOCHS}  val acc {acc:.3f}')
    if acc > best:
        best = acc
        best_state = {k: v.clone() for k,v in model.state_dict().items()}

model.load_state_dict(best_state)
acc, P, L = evaluate(test_dl)
print(f'\nTEST accuracy: {acc:.3f}\n')
print(classification_report(L, P, target_names=train_ds.classes, digits=3))
print(confusion_matrix(L, P))

SAVE = '/content/drive/MyDrive/Waste assistant'
torch.save(best_state, f'{SAVE}/waste_classifier.pth')
json.dump(train_ds.classes, open(f'{SAVE}/classes.json','w'))
print('saved to Drive')

device: cuda
['ewaste', 'glass', 'hazardous', 'metal', 'organic', 'paper_cardboard', 'plastic_bottle_container', 'plastic_film']
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 116MB/s]


epoch 1/12  val acc 0.740
epoch 2/12  val acc 0.825
epoch 3/12  val acc 0.881
epoch 4/12  val acc 0.929
epoch 5/12  val acc 0.903
epoch 6/12  val acc 0.933
epoch 7/12  val acc 0.929
epoch 8/12  val acc 0.941
epoch 9/12  val acc 0.941
epoch 10/12  val acc 0.941
epoch 11/12  val acc 0.941
epoch 12/12  val acc 0.941

TEST accuracy: 0.922

                          precision    recall  f1-score   support

                  ewaste      1.000     0.932     0.965        44
                   glass      0.889     0.889     0.889        45
               hazardous      0.935     0.967     0.951        30
                   metal      0.796     0.956     0.869        45
                 organic      1.000     1.000     1.000        25
         paper_cardboard      0.976     0.889     0.930        45
plastic_bottle_container      0.933     0.903     0.918        31
            plastic_film      1.000     0.500     0.667         4

                accuracy                          0.922       269


In [ ]:
import torch, executorch, torchao

print("PyTorch:", torch.__version__)
print("ExecuTorch:", executorch.__version__)
print("TorchAO:", torchao.__version__)

ModuleNotFoundError: No module named 'executorch'

In [ ]:
import torch
import torchvision
import importlib.util

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

print("ExecuTorch installed:", importlib.util.find_spec("executorch") is not None)
print("TorchAO installed:", importlib.util.find_spec("torchao") is not None)

PyTorch: 2.14.1+cpu
Torchvision: 0.29.1+cpu
ExecuTorch installed: False
TorchAO installed: True


In [ ]:
!pip install torchao==0.15.0 executorch==1.1.0

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 83.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 77.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 81.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 110.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 63.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 45.1 MB/s  0:00:00
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144590 sha256=de9304a165367660ef626712bec37c095f334f47c0c4860058a11947a3fbdda3
  Stored in directory: /root/.cache/pip/wheels/d5/b3/74/a35b66048c9de6631cd74cbc9475e6feb3e69a467983446bd8
Successfully built antlr4-python3-runtime
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Un

In [ ]:
import torch
import executorch
import torchao

print("PyTorch:", torch.__version__)
print("ExecuTorch:", executorch.__version__)
print("TorchAO:", torchao.__version__)

PyTorch: 2.14.1+cpu


AttributeError: module 'executorch' has no attribute '__version__'

In [ ]:
import importlib.util
import importlib.metadata
import torch
import torchao

print("PyTorch:", torch.__version__)
print("TorchAO:", torchao.__version__)

spec = importlib.util.find_spec("executorch")
print("ExecuTorch module found:", spec is not None)

try:
    print("ExecuTorch package version:",
          importlib.metadata.version("executorch"))
except importlib.metadata.PackageNotFoundError:
    print("ExecuTorch package metadata not found")


PyTorch: 2.14.1+cpu
TorchAO: 0.15.0
ExecuTorch module found: True
ExecuTorch package version: 1.1.0


In [ ]:
import torch
import executorch
from executorch.exir import to_edge

print("torch.export available:", hasattr(torch, "export"))
print("to_edge available:", callable(to_edge))

try:
    from executorch.exir import ExecutorchBackendConfig
    print("ExecuTorch export API: available")
except ImportError as e:
    print("Export API issue:", e)

torch.export available: True
to_edge available: True
ExecuTorch export API: available


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os

SAVE = "/content/drive/MyDrive/Waste assistant"

for filename in ["waste_classifier.pth", "classes.json"]:
    path = os.path.join(SAVE, filename)
    print(filename, "FOUND" if os.path.exists(path) else "MISSING")

waste_classifier.pth MISSING
classes.json MISSING


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os

SAVE = "/content/drive/MyDrive/Waste assistant"

for filename in ["waste_classifier.pth", "classes.json"]:
    path = os.path.join(SAVE, filename)
    print(filename, "FOUND" if os.path.exists(path) else "MISSING")

waste_classifier.pth FOUND
classes.json FOUND


In [ ]:
import torch
import torch.nn as nn
from executorch.exir import to_edge

model = nn.Linear(10, 9).eval()
x = torch.randn(1, 10)

try:
    exported = torch.export.export(model, (x,))
    edge = to_edge(exported)
    program = edge.to_executorch()

    print("EXPORT TEST PASSED")
    print("Test PTE size:", len(program.buffer), "bytes")

except Exception as e:
    print("EXPORT TEST FAILED")
    print(type(e).__name__, str(e))

EXPORT TEST PASSED
Test PTE size: 1940 bytes


/usr/local/lib/python3.13/dist-packages/executorch/exir/tensor.py:83: FutureWarning: guard_size_oblivious will be removed. Consider using explicit unbacked handling     potentially utilizing guard_or_false, guard_or_true, or statically_known_true
  return guard_size_oblivious(self.stride < other.stride)


In [ ]:
import os
import json
import torch
import torch.nn as nn
from torchvision import models
from executorch.exir import to_edge

SAVE = "/content/drive/MyDrive/Waste assistant"

PTH_PATH = os.path.join(SAVE, "waste_classifier.pth")
CLASSES_PATH = os.path.join(SAVE, "classes.json")
PTE_PATH = os.path.join(SAVE, "waste_classifier.pte")

# Load class names
with open(CLASSES_PATH, "r") as f:
    classes = json.load(f)

num_classes = len(classes)

print("Classes:", classes)
print("Number of classes:", num_classes)

# Rebuild MobileNetV3-Small
model = models.mobilenet_v3_small(weights=None)
model.classifier[3] = nn.Linear(
    model.classifier[3].in_features,
    num_classes
)

# Load your trained checkpoint
state_dict = torch.load(
    PTH_PATH,
    map_location="cpu",
    weights_only=True
)

model.load_state_dict(state_dict, strict=True)
model.eval()

print("Trained weights loaded successfully!")

# Export
example_input = torch.randn(1, 3, 224, 224)

exported_program = torch.export.export(
    model,
    (example_input,),
    strict=False
)

edge_program = to_edge(exported_program)
executorch_program = edge_program.to_executorch()

# Save PTE
with open(PTE_PATH, "wb") as f:
    f.write(executorch_program.buffer)

print("\nEXPORT SUCCESSFUL!")
print("Saved at:", PTE_PATH)
print("Size:", round(os.path.getsize(PTE_PATH) / (1024 * 1024), 2), "MB")

Classes: ['ewaste', 'glass', 'hazardous', 'metal', 'organic', 'paper_cardboard', 'plastic_bottle_container', 'plastic_film']
Number of classes: 8
Trained weights loaded successfully!


/usr/local/lib/python3.13/dist-packages/executorch/exir/tensor.py:83: FutureWarning: guard_size_oblivious will be removed. Consider using explicit unbacked handling     potentially utilizing guard_or_false, guard_or_true, or statically_known_true
  return guard_size_oblivious(self.stride < other.stride)



EXPORT SUCCESSFUL!
Saved at: /content/drive/MyDrive/Waste assistant/waste_classifier.pte
Size: 5.96 MB


In [ ]:
import os

path = "/content/drive/MyDrive/Waste assistant/waste_classifier.pte"

print("PTE exists:", os.path.exists(path))
print("PTE size:", os.path.getsize(path) if os.path.exists(path) else "Not found")

PTE exists: True
PTE size: 6250248


In [ ]:
from executorch.runtime import Runtime
import torch

runtime = Runtime.get()
program = runtime.load_program(
    "/content/drive/MyDrive/Waste assistant/waste_classifier.pte"
)

print("PTE loaded successfully!")

ImportError: /usr/local/lib/python3.13/dist-packages/executorch/extension/pybindings/_portable_lib.cpython-313-x86_64-linux-gnu.so: undefined symbol: _ZN3c104impl3cow23materialize_cow_storageERNS_11StorageImplE

In [ ]:
import os

path = "/content/drive/MyDrive/Waste assistant/waste_classifier.pte"

print("PTE exists:", os.path.exists(path))

if os.path.exists(path):
    print("PTE size:", os.path.getsize(path), "bytes")

PTE exists: True
PTE size: 6250248 bytes


In [ ]:
import sys
import torch
import importlib.metadata as metadata

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("ExecuTorch:", metadata.version("executorch"))
print("TorchAO:", metadata.version("torchao"))

Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
PyTorch: 2.14.1+cpu
ExecuTorch: 1.1.0
TorchAO: 0.15.0


In [ ]:
import os
import hashlib

path = "/content/drive/MyDrive/Waste assistant/waste_classifier.pte"

with open(path, "rb") as f:
    data = f.read()

print("File size:", len(data))
print("First 16 bytes:", data[:16].hex())
print("SHA256:", hashlib.sha256(data).hexdigest())

File size: 6250248
First 16 bytes: 3c000000455431326568303020000000
SHA256: f2a648a878d85592b0fbfb36f8c897af77e34748ad2bb2dbb335eae707292d89


In [20]:
!pip install executorch

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.3/25.3 MB 97.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 75.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 539.2/539.2 MB 40.8 MB/s  0:00:10
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 MB 18.9 MB/s  0:00:27
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 35.3 MB/s  0:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 48.9 MB/s  0:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 MB 42.9 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 52.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.9/404.9 MB 20.9 MB/s  0:00:14
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 36.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 MB 31.5 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━

In [21]:

import torch, json, torch.nn as nn
from torchvision import models
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner

SAVE = '/content/drive/MyDrive/Waste assistant'
classes = json.load(open(f'{SAVE}/classes.json'))
print(classes)

model = models.mobilenet_v3_small(weights=None)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, len(classes))
model.load_state_dict(torch.load(f'{SAVE}/waste_classifier.pth', map_location='cpu'))
model.eval()                      # important: inference mode

example = (torch.randn(1, 3, 224, 224),)
with torch.no_grad():
    ep = torch.export.export(model, example)
edge = to_edge_transform_and_lower(ep, partitioner=[XnnpackPartitioner()])
et = edge.to_executorch()
with open(f'{SAVE}/waste_classifier.pte', 'wb') as f:
    f.write(et.buffer)
print('saved', len(et.buffer) / 1e6, 'MB')

# Check: does the exported file give the same answer as the original model?
from executorch.runtime import Runtime
method = Runtime.get().load_program(f'{SAVE}/waste_classifier.pte').load_method('forward')
x = torch.randn(1, 3, 224, 224)
out = method.execute([x])[0]
with torch.no_grad():
    ref = model(x)
print('max difference:', (out - ref).abs().max().item())
print('original :', torch.softmax(ref, 1))
print('exported :', torch.softmax(out, 1))

ImportError: cannot import name 'fp32_matmul_precision_key' from 'torch._inductor.utils' (/usr/local/lib/python3.13/dist-packages/torch/_inductor/utils.py)

In [17]:
from PIL import Image
from torchvision import transforms
from executorch.runtime import Runtime
import torch

# Load the exact same image used in your earlier Colab test
img_path = '/content/drive/MyDrive/Waste assistant/test_ewaste.jpg'

ev = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

img = Image.open(img_path).convert("RGB")
x = ev(img).unsqueeze(0)

# Original PyTorch model
with torch.no_grad():
    original = model(x)

# Exported ExecuTorch model
method = Runtime.get().load_program(
    f'{SAVE}/waste_classifier.pte'
).load_method('forward')

exported = method.execute([x])[0]

# Compare predictions
p1 = torch.softmax(original, dim=1)[0]
p2 = torch.softmax(exported, dim=1)[0]

print("ORIGINAL MODEL:")
print(classes[p1.argmax().item()], p1.max().item() * 100)

print("\nEXPORTED MODEL:")
print(classes[p2.argmax().item()], p2.max().item() * 100)

print("\nMaximum difference:", (original - exported).abs().max().item())

ModuleNotFoundError: No module named 'executorch'

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls "/content/drive/MyDrive/Waste assistant"

 classes.json	        raw.zip			     waste_classifier.pte
 dataset.zip	        tokenizer.model		     waste_classifier.pth
 knowledge_base.json   ' waste assistant .ipynb'
 llama3_2_1b_v110.pte  'waste assistant new.ipynb'


In [ ]:
!ls dataset

ls: cannot access 'dataset': No such file or directory


In [ ]:
!rm -rf /content/raw /content/dataset

In [ ]:
!unzip -q -o "/content/drive/MyDrive/Waste assistant/dataset.zip" -d /content/
!ls /content

drive  raw  sample_data


In [ ]:
!mv /content/raw /content/dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls "/content/drive/MyDrive/Waste assistant"


Mounted at /content/drive
 classes.json	        raw.zip			     waste_classifier.pte
 dataset.zip	        tokenizer.model		     waste_classifier.pth
 knowledge_base.json   ' waste assistant .ipynb'
 llama3_2_1b_v110.pte  'waste assistant new.ipynb'


In [ ]:
!unzip -q -o "/content/drive/MyDrive/Waste assistant/dataset.zip" -d /content/
!ls /content

drive  raw  sample_data


In [ ]:
!ls /content/raw

biodegradable  ewaste  hazardous  recyclable


In [ ]:
!mv /content/raw /content/dataset

In [ ]:
# Colab cell: turns your 4-group folders into the 8 training classes (data8), cleaned and balanced.
import os, shutil, random, hashlib, re
from PIL import Image

SRC = "dataset"     # your 4 group folders: biodegradable, recyclable, e waste, hazardous
DST = "data8"       # the folder the training notebook should read
CAP = 400           # most images kept per class (None = keep all)
random.seed(42)

def norm(s):        # "E Waste", "e-waste", "ewaste" all become "ewaste"
    return re.sub(r"[^a-z0-9]", "", s.lower())

def to_class(parts):
    """parts = folder names from the group folder down to the image."""
    top = norm(parts[0]); rest = [norm(x) for x in parts[1:-1]]
    if top.startswith("bio"):  return "organic"
    if top.startswith(("ewaste", "electr", "waste")) or top == "e": return "ewaste"
    if top.startswith("haz"):  return "hazardous"
    if top.startswith("rec"):
        if any(x in ("cardboard", "paper") for x in rest): return "paper_cardboard"
        if "glass" in rest: return "glass"
        if "metal" in rest: return "metal"
        if "plastic" in rest:
            if any(x in ("packets", "packet", "film") for x in rest): return "plastic_film"
            return "plastic_bottle_container"          # PET and HDPE
    return None

found, unmapped, seen, bad, dup = {}, [], set(), 0, 0
for root, _, files in os.walk(SRC):
    for f in files:
        if not f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")): continue
        path = os.path.join(root, f)
        parts = os.path.relpath(path, SRC).split(os.sep)
        cls = to_class(parts)
        if cls is None: unmapped.append(path); continue
        try:
            with Image.open(path) as im: im.verify()         # skip broken files
        except Exception:
            bad += 1; continue
        h = hashlib.md5(open(path, "rb").read()).hexdigest()
        if h in seen: dup += 1; continue                    # skip exact duplicates
        seen.add(h)
        sub = "/".join(parts[1:-1])                         # sub-folder, used to balance
        found.setdefault(cls, {}).setdefault(sub, []).append(path)

shutil.rmtree(DST, ignore_errors=True)
for cls, subs in found.items():
    os.makedirs(f"{DST}/{cls}")
    for l in subs.values(): random.shuffle(l)
    chosen, i = [], 0
    while any(subs.values()) and (CAP is None or len(chosen) < CAP):   # take evenly from every sub-folder
        for l in subs.values():
            if l and (CAP is None or len(chosen) < CAP): chosen.append(l.pop())
    for k, p in enumerate(chosen):
        shutil.copy(p, f"{DST}/{cls}/{k}_{os.path.basename(p)}")

print("broken skipped:", bad, " duplicates skipped:", dup, " unmapped files:", len(unmapped))
for p in unmapped[:10]: print("  unmapped:", p)
for c in sorted(os.listdir(DST)): print(f"{c:28s}", len(os.listdir(f'{DST}/{c}')))

broken skipped: 0  duplicates skipped: 3  unmapped files: 0
ewaste                       299
glass                        300
hazardous                    196
metal                        300
organic                      164
paper_cardboard              297
plastic_bottle_container     212
plastic_film                 35


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls "/content/drive/MyDrive/Waste assistant"

 classes.json	        raw.zip			     waste_classifier.pte
 dataset.zip	        tokenizer.model		     waste_classifier.pth
 knowledge_base.json   ' waste assistant .ipynb'
 llama3_2_1b_v110.pte  'waste assistant new.ipynb'


In [ ]:
!unzip -q -o "/content/drive/MyDrive/Waste assistant/dataset.zip" -d /content/
!ls /content

drive  raw  sample_data


In [ ]:
!ls /content/raw

biodegradable  ewaste  hazardous  recyclable


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# Colab cell: turns your 4-group folders into the 8 training classes (data8), cleaned and balanced.
import os, shutil, random, hashlib, re
from PIL import Image

SRC = "dataset"     # your 4 group folders: biodegradable, recyclable, e waste, hazardous
DST = "data8"       # the folder the training notebook should read
CAP = 400           # most images kept per class (None = keep all)
random.seed(42)

def norm(s):        # "E Waste", "e-waste", "ewaste" all become "ewaste"
    return re.sub(r"[^a-z0-9]", "", s.lower())

def to_class(parts):
    """parts = folder names from the group folder down to the image."""
    top = norm(parts[0]); rest = [norm(x) for x in parts[1:-1]]
    if top.startswith("bio"):  return "organic"
    if top.startswith(("ewaste", "electr", "waste")) or top == "e": return "ewaste"
    if top.startswith("haz"):  return "hazardous"
    if top.startswith("rec"):
        if any(x in ("cardboard", "paper") for x in rest): return "paper_cardboard"
        if "glass" in rest: return "glass"
        if "metal" in rest: return "metal"
        if "plastic" in rest:
            if any(x in ("packets", "packet", "film") for x in rest): return "plastic_film"
            return "plastic_bottle_container"          # PET and HDPE
    return None

found, unmapped, seen, bad, dup = {}, [], set(), 0, 0
for root, _, files in os.walk(SRC):
    for f in files:
        if not f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")): continue
        path = os.path.join(root, f)
        parts = os.path.relpath(path, SRC).split(os.sep)
        cls = to_class(parts)
        if cls is None: unmapped.append(path); continue
        try:
            with Image.open(path) as im: im.verify()         # skip broken files
        except Exception:
            bad += 1; continue
        h = hashlib.md5(open(path, "rb").read()).hexdigest()
        if h in seen: dup += 1; continue                    # skip exact duplicates
        seen.add(h)
        sub = "/".join(parts[1:-1])                         # sub-folder, used to balance
        found.setdefault(cls, {}).setdefault(sub, []).append(path)

shutil.rmtree(DST, ignore_errors=True)
for cls, subs in found.items():
    os.makedirs(f"{DST}/{cls}")
    for l in subs.values(): random.shuffle(l)
    chosen, i = [], 0
    while any(subs.values()) and (CAP is None or len(chosen) < CAP):   # take evenly from every sub-folder
        for l in subs.values():
            if l and (CAP is None or len(chosen) < CAP): chosen.append(l.pop())
    for k, p in enumerate(chosen):
        shutil.copy(p, f"{DST}/{cls}/{k}_{os.path.basename(p)}")

print("broken skipped:", bad, " duplicates skipped:", dup, " unmapped files:", len(unmapped))
for p in unmapped[:10]: print("  unmapped:", p)
for c in sorted(os.listdir(DST)): print(f"{c:28s}", len(os.listdir(f'{DST}/{c}')))

broken skipped: 0  duplicates skipped: 3  unmapped files: 0
ewaste                       299
glass                        300
hazardous                    196
metal                        300
organic                      164
paper_cardboard              297
plastic_bottle_container     212
plastic_film                 283


In [10]:
import torch, torch.nn as nn, numpy as np, json
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix

OUT = 'data8'
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev)

mean, std = [0.485,0.456,0.406], [0.229,0.224,0.225]
tr = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.3,0.3,0.3),
    transforms.ToTensor(), transforms.Normalize(mean,std)])
ev = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(), transforms.Normalize(mean,std)])

# Load the entire dataset from the processed data8 directory
full_dataset = datasets.ImageFolder(OUT)
K = len(full_dataset.classes)
print(full_dataset.classes)

# Set random seed for reproducibility
torch.manual_seed(42)

# Split into train (70%), validation (15%), and test (15%)
n_total = len(full_dataset)
n_train = int(0.70 * n_total)
n_val = int(0.15 * n_total)
n_test = n_total - n_train - n_val

train_ds, val_ds, test_ds = torch.utils.data.random_split(full_dataset, [n_train, n_val, n_test])

# Override transforms for splits
train_ds.dataset.transform = tr
val_ds.dataset.transform = ev
test_ds.dataset.transform = ev

train_dl = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_dl   = torch.utils.data.DataLoader(val_ds, batch_size=64, num_workers=2)
test_dl  = torch.utils.data.DataLoader(test_ds, batch_size=64, num_workers=2)

# Class weights computed from training targets
train_targets = [full_dataset.targets[i] for i in train_ds.indices]
cnt = np.bincount(train_targets, minlength=K)
w = torch.tensor(len(train_ds)/(K*cnt), dtype=torch.float).to(dev)

model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, K)
model.to(dev)

EPOCHS = 12
opt = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
loss_fn = nn.CrossEntropyLoss(weight=w)

def evaluate(dl):
    model.eval(); P=[]; L=[]
    with torch.no_grad():
        for x,y in dl:
            P += model(x.to(dev)).argmax(1).cpu().tolist(); L += y.tolist()
    return np.mean(np.array(P)==np.array(L)), P, L

best, best_state = 0, None
for ep in range(EPOCHS):
    model.train()
    for x,y in train_dl:
        x,y = x.to(dev), y.to(dev)
        opt.zero_grad(); loss_fn(model(x), y).backward(); opt.step()
    sched.step()
    acc,_,_ = evaluate(val_dl)
    print(f'epoch {ep+1}/{EPOCHS}  val acc {acc:.3f}')
    if acc > best:
        best = acc
        best_state = {k: v.clone() for k,v in model.state_dict().items()}

model.load_state_dict(best_state)
acc, P, L = evaluate(test_dl)
print(f'\nTEST accuracy: {acc:.3f}\n')
print(classification_report(L, P, target_names=full_dataset.classes, digits=3))
print(confusion_matrix(L, P))

SAVE = '/content/drive/MyDrive/Waste assistant'
os.makedirs(SAVE, exist_ok=True)
torch.save(best_state, f'{SAVE}/waste_classifier1.pth')
json.dump(full_dataset.classes, open(f'{SAVE}/classes1.json','w'))
print('saved to Drive')

device: cuda
['ewaste', 'glass', 'hazardous', 'metal', 'organic', 'paper_cardboard', 'plastic_bottle_container', 'plastic_film']
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 90.6MB/s]


epoch 1/12  val acc 0.798
epoch 2/12  val acc 0.870
epoch 3/12  val acc 0.938
epoch 4/12  val acc 0.922
epoch 5/12  val acc 0.915
epoch 6/12  val acc 0.951
epoch 7/12  val acc 0.941
epoch 8/12  val acc 0.935
epoch 9/12  val acc 0.954
epoch 10/12  val acc 0.951
epoch 11/12  val acc 0.951
epoch 12/12  val acc 0.954

TEST accuracy: 0.958

                          precision    recall  f1-score   support

                  ewaste      1.000     0.958     0.979        48
                   glass      0.941     0.906     0.923        53
               hazardous      0.889     0.960     0.923        25
                   metal      0.915     0.956     0.935        45
                 organic      0.962     1.000     0.980        25
         paper_cardboard      1.000     0.944     0.971        36
plastic_bottle_container      0.964     1.000     0.982        27
            plastic_film      0.980     0.980     0.980        50

                accuracy                          0.958       309


In [2]:
import os
import torch
import json
from torchvision import models
import torch.nn as nn

SAVE = "/content/drive/MyDrive/Waste assistant"
os.makedirs(SAVE, exist_ok=True)

# Re-define classes and number of classes from your processed dataset
classes = ['ewaste', 'glass', 'hazardous', 'metal', 'organic', 'paper_cardboard', 'plastic_bottle_container', 'plastic_film']
num_classes = len(classes)

# Rebuild the model architecture
model = models.mobilenet_v3_small(weights=None)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)

# Load the best state dict that was saved during training
checkpoint_path = f"{SAVE}/waste_classifier1.pth"
if os.path.exists(checkpoint_path):
    state_dict = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(state_dict)
    print("Loaded trained weights from Google Drive.")
else:
    print(f"Warning: {checkpoint_path} not found. Saving an uninitialized model template instead.")

# Save trained model weights to the final target file
torch.save(
    model.state_dict(),
    f"{SAVE}/waste_classifier.pth"
)

# Save class labels in the correct order
with open(f"{SAVE}/classes.json", "w") as f:
    json.dump(classes, f)

print("Model and class labels saved successfully!")

Model and class labels saved successfully!


In [4]:
# Install executorch and torchao matching dependencies if they are missing in this session
import sys
try:
    import executorch
except ImportError:
    !{sys.executable} -m pip install -q "torchao==0.15.0" "executorch==1.1.0"

import torch
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import (
    XnnpackPartitioner
)

model.eval()
model.cpu()

example_inputs = (torch.randn(1, 3, 224, 224),)

exported_program = torch.export.export(
    model,
    example_inputs
)

edge_program = to_edge_transform_and_lower(
    exported_program,
    partitioner=[XnnpackPartitioner()]
)

executorch_program = edge_program.to_executorch()

pte_path = f"{SAVE}/waste_classifier new.pte"

with open(pte_path, "wb") as f:
    f.write(executorch_program.buffer)

print("ExecuTorch model exported:", pte_path)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


ExecuTorch model exported: /content/drive/MyDrive/Waste assistant/waste_classifier new.pte


/usr/local/lib/python3.13/dist-packages/executorch/exir/tensor.py:83: FutureWarning: guard_size_oblivious will be removed. Consider using explicit unbacked handling     potentially utilizing guard_or_false, guard_or_true, or statically_known_true
  return guard_size_oblivious(self.stride < other.stride)


In [8]:
import os
import json

SAVE = "/content/drive/MyDrive/Waste assistant"
classes = ['ewaste', 'glass', 'hazardous', 'metal', 'organic', 'paper_cardboard', 'plastic_bottle_container', 'plastic_film']

# Save classes new.json as requested
classes_new_path = os.path.join(SAVE, "classes new.json")
with open(classes_new_path, "w") as f:
    json.dump(classes, f)
print("Saved:", classes_new_path)

# Ensure the PTE file is copied to any alternative names if needed, and flush to Drive
pte_source = os.path.join(SAVE, "waste_classifier new.pte")
pte_alt = os.path.join(SAVE, "waste_classifier.pte")

if os.path.exists(pte_source):
    print("Found source PTE:", pte_source)
    # Copy to waste_classifier.pte as well, in case your application is looking for that specific name
    import shutil
    shutil.copy2(pte_source, pte_alt)
    print("Synchronized copy to:", pte_alt)

# Force Google Drive to sync instantly
try:
    from google.colab import drive
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    print("Drive flushed and remounted successfully to guarantee files are visible.")
except Exception as e:
    print("Could not flush Drive automatically, you can manually check your Drive folder now:", e)

Saved: /content/drive/MyDrive/Waste assistant/classes new.json
Found source PTE: /content/drive/MyDrive/Waste assistant/waste_classifier new.pte
Synchronized copy to: /content/drive/MyDrive/Waste assistant/waste_classifier.pte
Drive not mounted, so nothing to flush and unmount.
Could not flush Drive automatically, you can manually check your Drive folder now: Mountpoint must not already contain files


In [9]:
import os
import json
import torch
import torch.nn as nn
from torchvision import models

SAVE = "/content/drive/MyDrive/Waste assistant"
os.makedirs(SAVE, exist_ok=True)

# 1. Create and save "classes new.json"
classes = ['ewaste', 'glass', 'hazardous', 'metal', 'organic', 'paper_cardboard', 'plastic_bottle_container', 'plastic_film']
classes_new_path = os.path.join(SAVE, "classes new.json")
with open(classes_new_path, "w") as f:
    json.dump(classes, f)
print("Saved:", classes_new_path)

# 2. Copy the checkpoint to "waste_classifier new.pth"
pth_source = os.path.join(SAVE, "waste_classifier.pth")
pth_new = os.path.join(SAVE, "waste_classifier new.pth")

if os.path.exists(pth_source):
    import shutil
    shutil.copy2(pth_source, pth_new)
    print("Saved:", pth_new)
else:
    print(f"Warning: {pth_source} not found. Please make sure training has run.")

# 3. Export ExecuTorch model to "waste_classifier new.pte"
try:
    from executorch.exir import to_edge_transform_and_lower
    from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner

    # Rebuild MobileNetV3-Small architecture
    model = models.mobilenet_v3_small(weights=None)
    model.classifier[3] = nn.Linear(model.classifier[3].in_features, len(classes))

    # Load trained state dictionary
    state_dict = torch.load(pth_new, map_location="cpu")
    model.load_state_dict(state_dict)
    model.eval()

    # Run export compilation
    example_inputs = (torch.randn(1, 3, 224, 224),)
    exported_program = torch.export.export(model, example_inputs)
    edge_program = to_edge_transform_and_lower(exported_program, partitioner=[XnnpackPartitioner()])
    executorch_program = edge_program.to_executorch()

    pte_path = os.path.join(SAVE, "waste_classifier new.pte")
    with open(pte_path, "wb") as f:
        f.write(executorch_program.buffer)

    print("Exported and saved:", pte_path)
except Exception as e:
    print("ExecuTorch export failed:", e)

# 4. Flush Google Drive to make files instantly visible
try:
    from google.colab import drive
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    print("Drive flushed successfully! Check your Drive now.")
except Exception as e:
    print("Could not flush Drive automatically:", e)

Saved: /content/drive/MyDrive/Waste assistant/classes new.json
Saved: /content/drive/MyDrive/Waste assistant/waste_classifier new.pth


/usr/local/lib/python3.13/dist-packages/executorch/exir/tensor.py:83: FutureWarning: guard_size_oblivious will be removed. Consider using explicit unbacked handling     potentially utilizing guard_or_false, guard_or_true, or statically_known_true
  return guard_size_oblivious(self.stride < other.stride)


Exported and saved: /content/drive/MyDrive/Waste assistant/waste_classifier new.pte
Drive not mounted, so nothing to flush and unmount.
Could not flush Drive automatically: Mountpoint must not already contain files


In [10]:
import os
from google.colab import files

pte_file_path = "/content/drive/MyDrive/Waste assistant/waste_classifier new.pte"

if os.path.exists(pte_file_path):
    print(f"File found (size: {os.path.getsize(pte_file_path) / (1024*1024):.2f} MB). Triggering browser download...")
    files.download(pte_file_path)
else:
    print("Could not locate the .pte file at the path. Please check if the previous export cell finished running successfully.")

File found (size: 5.83 MB). Triggering browser download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
import json

In [6]:
SAVE = '/content/drive/MyDrive/Waste assistant'

MODEL_PATH = f'{SAVE}/waste_classifier_new.pth'
CLASSES_PATH = f'{SAVE}/classes_new.json'

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
import os
import json
import torch
import torch.nn as nn
from torchvision import models

SAVE = '/content/drive/MyDrive/Waste assistant'

MODEL_PATH = f'{SAVE}/waste_classifier.pth'
CLASSES_PATH = f'{SAVE}/classes.json'

print("Folder exists:", os.path.exists(SAVE))

if os.path.exists(SAVE):
    print("Files:", os.listdir(SAVE))

Folder exists: True
Files: ['waste assistant new.ipynb', 'classes.json', 'waste_classifier.pth', 'waste_classifier.pte', 'tokenizer.model', ' waste assistant .ipynb', 'llama3_2_1b_v110.pte', 'raw.zip', 'knowledge_base.json', 'dataset.zip']


In [10]:
assert os.path.isfile(MODEL_PATH), "waste_classifier.pth not found"
assert os.path.isfile(CLASSES_PATH), "classes.json not found"

classes = json.load(open(CLASSES_PATH))

print("Classes:", classes)
print("Number of classes:", len(classes))

Classes: ['ewaste', 'glass', 'hazardous', 'metal', 'organic', 'paper_cardboard', 'plastic_bottle_container', 'plastic_film']
Number of classes: 8


In [11]:
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

model = models.mobilenet_v3_small(weights=None)

model.classifier[3] = nn.Linear(
    model.classifier[3].in_features,
    len(classes)
)

state = torch.load(
    MODEL_PATH,
    map_location=dev,
    weights_only=True
)

model.load_state_dict(state)
model.to(dev)
model.eval()

print("Model loaded successfully!")

Model loaded successfully!


In [12]:
img_path = '/content/drive/MyDrive/Waste assistant/test_ewaste.jpg'

In [13]:
import os

print(os.path.exists(img_path))

True


In [16]:
from torchvision import transforms
from PIL import Image
import torch

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

ev = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

img = Image.open(img_path).convert("RGB")
x = ev(img).unsqueeze(0).to(dev)

with torch.no_grad():
    logits = model(x)
    probs = torch.softmax(logits, dim=1)[0]

for i in torch.argsort(probs, descending=True):
    print(f"{classes[i]}: {probs[i].item()*100:.2f}%")

ewaste: 98.33%
hazardous: 1.22%
paper_cardboard: 0.19%
metal: 0.17%
plastic_bottle_container: 0.07%
plastic_film: 0.02%
glass: 0.00%
organic: 0.00%


In [22]:
import torch
import sys

print("Python:", sys.version)
print("PyTorch:", torch.__version__)

Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
PyTorch: 2.14.1+cpu


In [23]:
import importlib.metadata as md

for package in ["executorch", "torch", "torchvision"]:
    try:
        print(package, md.version(package))
    except Exception as e:
        print(package, "Not found:", e)

executorch 1.6.0
torch 2.15.0
torchvision 0.29.1+cpu


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import torch
import json
import torch.nn as nn
from torchvision import models

from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.runtime import Runtime

print("PyTorch:", torch.__version__)
print("All imports successful!")

W1125 06:09:57.036000 10554 torch/utils/_pytree.py:646] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


PyTorch: 2.15.0+cu132
All imports successful!


In [3]:
SAVE = '/content/drive/MyDrive/Waste assistant'

classes = json.load(open(f'{SAVE}/classes.json'))

model = models.mobilenet_v3_small(weights=None)
model.classifier[3] = nn.Linear(
    model.classifier[3].in_features,
    len(classes)
)

state = torch.load(
    f'{SAVE}/waste_classifier.pth',
    map_location='cpu',
    weights_only=True
)

model.load_state_dict(state)
model.eval()

print("Model loaded successfully!")
print("Classes:", classes)

Model loaded successfully!
Classes: ['ewaste', 'glass', 'hazardous', 'metal', 'organic', 'paper_cardboard', 'plastic_bottle_container', 'plastic_film']


In [4]:
import torch
import executorch

from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.runtime import Runtime

print("PyTorch:", torch.__version__)
print("ExecuTorch imports successful!")

PyTorch: 2.15.0+cu132
ExecuTorch imports successful!


In [5]:
from PIL import Image
from torchvision import transforms
import torch

img_path = f'{SAVE}/test_ewaste.jpg'

ev = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

img = Image.open(img_path).convert("RGB")
x = ev(img).unsqueeze(0)

# Original PyTorch model
with torch.no_grad():
    original = model(x)

# Exported ExecuTorch model
program = Runtime.get().load_program(f'{SAVE}/waste_classifier.pte')
method = program.load_method('forward')
exported = method.execute([x])[0]

# Compare predictions
p1 = torch.softmax(original, dim=1)[0]
p2 = torch.softmax(exported, dim=1)[0]

print("ORIGINAL .PTH MODEL:")
print(classes[p1.argmax().item()], f"{p1.max().item()*100:.2f}%")

print("\nEXPORTED .PTE MODEL:")
print(classes[p2.argmax().item()], f"{p2.max().item()*100:.2f}%")

print("\nMaximum difference:",
      (original - exported).abs().max().item())

ORIGINAL .PTH MODEL:
ewaste 98.33%

EXPORTED .PTE MODEL:
paper_cardboard 12.81%

Maximum difference: 5.656862258911133


In [6]:
# 1. Export using the actual e-waste image tensor
ep = torch.export.export(model, (x,))

# 2. Check the captured PyTorch graph
captured = ep.module()(x)

print("Captured graph difference:",
      (original - captured).abs().max().item())

# 3. Lower WITHOUT XNNPACK
edge = to_edge_transform_and_lower(ep)

et = edge.to_executorch()

diagnostic_path = f'{SAVE}/diagnostic_portable.pte'

with open(diagnostic_path, 'wb') as f:
    f.write(et.buffer)

# 4. Run the portable ExecuTorch model
program = Runtime.get().load_program(diagnostic_path)
method = program.load_method('forward')
portable = method.execute([x])[0]

p = torch.softmax(portable, dim=1)[0]

print("\nPORTABLE EXECUTORCH MODEL:")
print(classes[p.argmax().item()], f"{p.max().item()*100:.2f}%")

print("\nPortable model difference:",
      (original - portable).abs().max().item())

Captured graph difference: 0.0

PORTABLE EXECUTORCH MODEL:
ewaste 98.33%

Portable model difference: 1.239776611328125e-05


In [7]:
ep_xnn = torch.export.export(model, (x.contiguous(),))

edge_xnn = to_edge_transform_and_lower(
    ep_xnn,
    partitioner=[XnnpackPartitioner()]
)

et_xnn = edge_xnn.to_executorch()

xnn_path = f'{SAVE}/diagnostic_xnnpack.pte'

with open(xnn_path, 'wb') as f:
    f.write(et_xnn.buffer)

program_xnn = Runtime.get().load_program(xnn_path)
method_xnn = program_xnn.load_method('forward')

xnn_output = method_xnn.execute([x.contiguous()])[0]

p = torch.softmax(xnn_output, dim=1)[0]

print("XNNPACK prediction:", classes[p.argmax().item()])
print("Confidence:", p.max().item() * 100)
print("Maximum difference:",
      (original - xnn_output).abs().max().item())

XNNPACK prediction: ewaste
Confidence: 98.33008050918579
Maximum difference: 9.5367431640625e-06


In [8]:
import os
import hashlib

for name in [
    "waste_classifier.pte",
    "diagnostic_portable.pte",
    "diagnostic_xnnpack.pte"
]:
    path = f"{SAVE}/{name}"
    with open(path, "rb") as f:
        digest = hashlib.sha256(f.read()).hexdigest()

    print(name)
    print("Size:", os.path.getsize(path))
    print("SHA256:", digest)
    print()

waste_classifier.pte
Size: 6115488
SHA256: 35c255d6600f6a8710201dde7fcd0cf9962144cc688e2cc2e979b1023b707939

diagnostic_portable.pte
Size: 6226312
SHA256: 3861ec3dcfcf610e427d2b24253bb63b7f595a6fc77cf1e7333aeebf5be32b08

diagnostic_xnnpack.pte
Size: 6137632
SHA256: 2f828bec854f5398a7d265dffdc3bb5b66accff2da8de5966e2bd86f03bf6485



In [6]:
!find /content /content/drive -name "classes.json" 2>/dev/null


/content/drive/MyDrive/Waste assistant/classes.json
/content/drive/MyDrive/backup/classes.json
/content/drive/MyDrive/Waste assistant/classes.json
/content/drive/MyDrive/backup/classes.json


In [7]:
import json

with open('/content/drive/MyDrive/Waste assistant/classes.json', 'r') as f:
    classes = json.load(f)

print("Class order:", classes)

Class order: ['ewaste', 'glass', 'hazardous', 'metal', 'organic', 'paper_cardboard', 'plastic_bottle_container', 'plastic_film']


In [10]:
!pip install executorch==1.6.0 torch==2.15.0 \
--index-url https://download.pytorch.org/whl/cpu \
--extra-index-url https://pypi.org/simple

Looking in indexes: https://download.pytorch.org/whl/cpu, https://pypi.org/simple
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.7/17.7 MB 112.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.5/209.5 MB 70.2 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 44.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 108.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 57.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 60.3 MB/s  0:00:00
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144590 sha256=1ab3746797db1361667dd20b4cb90e358386ab8251eb2b5b0562c2d327ff56cc
  Stored in directory: /root/.cache/pip/wheels/d5/b3/74/a35b66048c9de6631cd74cbc9475e6feb3e69a467983446bd8
Successfully built antlr4-python3-runtime

In [12]:
from PIL import Image

In [15]:
import torch
print("PyTorch:", torch.__version__)

import importlib.metadata as metadata

for package in ["executorch", "torch", "torchvision", "torchaudio"]:
    try:
        print(package, metadata.version(package))
    except metadata.PackageNotFoundError:
        print(package, "Not installed")

PyTorch: 2.14.1+cpu
executorch 1.6.0+cpu
torch 2.15.0+cpu
torchvision 0.29.1+cpu
torchaudio 2.11.0+cpu


In [2]:
!pip install --no-cache-dir executorch==1.6.0

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.3/25.3 MB 65.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 309.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 539.2/539.2 MB 222.5 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 MB 241.9 MB/s  0:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 MB 334.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.9/404.9 MB 92.4 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 256.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 MB 282.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 227.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 MB 367.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 480.8 MB/s  0:00:00
   ━━━━━━━━━━━━━

In [4]:
import torch
from importlib.metadata import version, PackageNotFoundError

print("PyTorch:", torch.__version__)

try:
    print("ExecuTorch:", version("executorch"))
except PackageNotFoundError:
    print("ExecuTorch package metadata not found")

PyTorch: 2.14.1+cu130
ExecuTorch: 1.6.0


In [1]:
!pip install --no-cache-dir executorch==1.6.0 torch==2.15.0 --index-url https://download.pytorch.org/whl/cu132 --extra-index-url https://pypi.org/simple

Looking in indexes: https://download.pytorch.org/whl/cu132, https://pypi.org/simple
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.2/25.2 MB 39.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 539.2/539.2 MB 157.5 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 92.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 MB 41.0 MB/s  0:00:09
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 46.8 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 39.2 MB/s  0:00:05
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 MB 21.7 MB/s  0:00:06
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 51.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.9/404.9 MB 74.8 MB/s  0:00:05
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 70.7 MB/s  0:00:00
   ━━━━━━━━━

In [2]:
import torch
from importlib.metadata import version, PackageNotFoundError

print("PyTorch:", torch.__version__)

try:
    print("ExecuTorch:", version("executorch"))
except PackageNotFoundError:
    print("ExecuTorch package metadata not found")

PyTorch: 2.15.0+cu132
ExecuTorch: 1.6.0+cu132


In [4]:
import os
import json
import torch
import torch.nn as nn
import torchvision
import executorch

from PIL import Image
from torchvision import transforms, models
from executorch.runtime import Runtime

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("ExecuTorch:", executorch.__version__ if hasattr(executorch, "__version__") else "Installed")
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")

PyTorch: 2.15.0+cu132
Torchvision: 0.29.1+cpu
ExecuTorch: Installed
Device: CPU


In [5]:
BASE = "/content/drive/MyDrive/Waste assistant"

PTH_PATH = f"{BASE}/waste_classifier.pth"
PTE_PATH = f"{BASE}/waste_classifier.pte"
CLASSES_PATH = f"{BASE}/classes.json"
IMAGE_PATH = f"{BASE}/test_ewaste.jpg"

for path in [PTH_PATH, PTE_PATH, CLASSES_PATH, IMAGE_PATH]:
    print(path, "EXISTS" if os.path.exists(path) else "MISSING")

/content/drive/MyDrive/Waste assistant/waste_classifier.pth MISSING
/content/drive/MyDrive/Waste assistant/waste_classifier.pte MISSING
/content/drive/MyDrive/Waste assistant/classes.json MISSING
/content/drive/MyDrive/Waste assistant/test_ewaste.jpg MISSING


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
BASE = "/content/drive/MyDrive/Waste assistant"

PTH_PATH = f"{BASE}/waste_classifier.pth"
PTE_PATH = f"{BASE}/waste_classifier.pte"
CLASSES_PATH = f"{BASE}/classes.json"
IMAGE_PATH = f"{BASE}/test_ewaste.jpg"

for path in [PTH_PATH, PTE_PATH, CLASSES_PATH, IMAGE_PATH]:
    print(path, "EXISTS" if os.path.exists(path) else "MISSING")

/content/drive/MyDrive/Waste assistant/waste_classifier.pth EXISTS
/content/drive/MyDrive/Waste assistant/waste_classifier.pte EXISTS
/content/drive/MyDrive/Waste assistant/classes.json EXISTS
/content/drive/MyDrive/Waste assistant/test_ewaste.jpg EXISTS


In [8]:
with open(CLASSES_PATH, "r") as f:
    classes = json.load(f)

print("Classes:", classes)
print("Number of classes:", len(classes))

Classes: ['ewaste', 'glass', 'hazardous', 'metal', 'organic', 'paper_cardboard', 'plastic_bottle_container', 'plastic_film']
Number of classes: 8


In [9]:
ev = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

img = Image.open(IMAGE_PATH).convert("RGB")
x = ev(img).unsqueeze(0)

print("Image:", img.size)
print("Input tensor:", x.shape)

Image: (1856, 4096)
Input tensor: torch.Size([1, 3, 224, 224])


In [10]:
model = models.mobilenet_v3_small(weights=None)

model.classifier[3] = nn.Linear(
    model.classifier[3].in_features,
    len(classes)
)

checkpoint = torch.load(PTH_PATH, map_location="cpu")

if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    checkpoint = checkpoint["state_dict"]

if any(k.startswith("module.") for k in checkpoint.keys()):
    checkpoint = {
        k.replace("module.", "", 1): v
        for k, v in checkpoint.items()
    }

model.load_state_dict(checkpoint)
model.eval()

print("Original PyTorch model loaded successfully.")

Original PyTorch model loaded successfully.


In [11]:
with torch.no_grad():
    original = model(x)

p1 = torch.softmax(original, dim=1)[0]

print("ORIGINAL PYTORCH MODEL")
for i in torch.argsort(p1, descending=True):
    print(f"{classes[i]}: {p1[i].item()*100:.2f}%")

print("\nPrediction:",
      classes[p1.argmax().item()],
      f"{p1.max().item()*100:.2f}%")

ORIGINAL PYTORCH MODEL
ewaste: 98.33%
hazardous: 1.22%
paper_cardboard: 0.19%
metal: 0.17%
plastic_bottle_container: 0.07%
plastic_film: 0.02%
glass: 0.00%
organic: 0.00%

Prediction: ewaste 98.33%


In [12]:
program = Runtime.get().load_program(PTE_PATH)
method = program.load_method("forward")

exported_output = method.execute([x])[0]

exported = torch.tensor(
    exported_output.numpy()
).reshape(1, -1)

p2 = torch.softmax(exported, dim=1)[0]

print("EXPORTED EXECUTORCH MODEL")
for i in torch.argsort(p2, descending=True):
    print(f"{classes[i]}: {p2[i].item()*100:.2f}%")

print("\nPrediction:",
      classes[p2.argmax().item()],
      f"{p2.max().item()*100:.2f}%")

EXPORTED EXECUTORCH MODEL
paper_cardboard: 12.81%
glass: 12.77%
plastic_bottle_container: 12.55%
ewaste: 12.53%
organic: 12.50%
hazardous: 12.38%
metal: 12.29%
plastic_film: 12.16%

Prediction: paper_cardboard 12.81%


In [13]:
print("Original logits:")
print(original)

print("\nExported output:")
print(exported)

print("\nMaximum absolute difference:")
print((original - exported).abs().max().item())

print("\nOriginal shape:", original.shape)
print("Exported shape:", exported.shape)

Original logits:
tensor([[ 5.6580, -4.3648,  1.2692, -0.6869, -4.6761, -0.6165, -1.6328, -3.0285]])

Exported output:
tensor([[ 0.0011,  0.0201, -0.0106, -0.0181, -0.0012,  0.0230,  0.0029, -0.0286]])

Maximum absolute difference:
5.656862258911133

Original shape: torch.Size([1, 8])
Exported shape: torch.Size([1, 8])


In [14]:
model.eval()
x = x.cpu().contiguous()

with torch.no_grad():
    original = model(x)

print("Original prediction:",
      classes[original.argmax(dim=1).item()])

Original prediction: ewaste


In [20]:
ep_xnn = torch.export.export(model, (x.contiguous(),))

edge_xnn = to_edge_transform_and_lower(
    ep_xnn,
    partitioner=[XnnpackPartitioner()]
)

et_xnn = edge_xnn.to_executorch()

xnn_path = f'{SAVE}/diagnostic_xnnpack.pte'

with open(xnn_path, 'wb') as f:
    f.write(et_xnn.buffer)

program_xnn = Runtime.get().load_program(xnn_path)
method_xnn = program_xnn.load_method('forward')

xnn_output = method_xnn.execute([x.contiguous()])[0]

p = torch.softmax(xnn_output, dim=1)[0]

print("XNNPACK prediction:", classes[p.argmax().item()])
print("Confidence:", p.max().item() * 100)
print("Maximum difference:",
      (original - xnn_output).abs().max().item())

XNNPACK prediction: ewaste
Confidence: 98.33008050918579
Maximum difference: 9.5367431640625e-06


In [16]:
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner

W1125 09:59:17.529000 1044 torch/utils/_pytree.py:646] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [18]:
SAVE = "/content/drive/MyDrive/Waste assistant"

import os
os.makedirs(SAVE, exist_ok=True)

print("SAVE =", SAVE)

SAVE = /content/drive/MyDrive/Waste assistant


In [21]:
import shutil

FINAL_PATH = f"{SAVE}/waste_classifier.pte"

shutil.copy2(
    f"{SAVE}/diagnostic_xnnpack.pte",
    FINAL_PATH
)

print("Updated model:", FINAL_PATH)


Updated model: /content/drive/MyDrive/Waste assistant/waste_classifier.pte


In [22]:
values = x.flatten()

print("Input stats:")
print("Min:", values.min().item())
print("Max:", values.max().item())
print("Mean:", values.mean().item())
print("First 15:", values[:15].tolist())

Input stats:
Min: -1.878157377243042
Max: 2.4482791423797607
Mean: -0.02087908796966076
First 15: [0.7248052358627319, 0.7761794924736023, 0.7248052358627319, 0.6563062071800232, 0.7419300079345703, 0.9303022623062134, 1.1186745166778564, 1.18717360496521, 1.2727973461151123, 1.2214230298995972, 1.18717360496521, 1.18717360496521, 1.3584210872650146, 1.5467933416366577, 1.4954191446304321]


In [23]:
print("Original image dimensions:", img.size)
print("Resized tensor:", x.shape)


Original image dimensions: (1856, 4096)
Resized tensor: torch.Size([1, 3, 224, 224])


In [26]:
np.save("/content/colab_input.npy", x.cpu().numpy())

In [25]:
import numpy as np
from PIL import Image
import torchvision.transforms as transforms


In [27]:
import numpy as np
from PIL import Image
import torchvision.transforms as transforms

# Load the exact same test image
image_path = f"{BASE}/test_ewaste.jpg"
img = Image.open(image_path).convert("RGB")

# Reproduce the model preprocessing
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

x = preprocess(img).unsqueeze(0)

# Save normalized tensor
np.save("/content/colab_input.npy", x.numpy())

# Save resized RGB image
resized = img.resize((224, 224), Image.Resampling.BILINEAR)
resized.save("/content/colab_resized.png")

# Print statistics
print("Tensor shape:", x.shape)
print("Min:", x.min().item())
print("Max:", x.max().item())
print("Mean:", x.mean().item())
print("First 15 values:", x.flatten()[:15].tolist())

print("Files saved successfully!")

Tensor shape: torch.Size([1, 3, 224, 224])
Min: -1.878157377243042
Max: 2.4482791423797607
Mean: -0.02087908796966076
First 15 values: [0.7248052358627319, 0.7761794924736023, 0.7248052358627319, 0.6563062071800232, 0.7419300079345703, 0.9303022623062134, 1.1186745166778564, 1.18717360496521, 1.2727973461151123, 1.2214230298995972, 1.18717360496521, 1.18717360496521, 1.3584210872650146, 1.5467933416366577, 1.4954191446304321]
Files saved successfully!


In [28]:
from google.colab import files

files.download("/content/colab_input.npy")
files.download("/content/colab_resized.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
resized = transforms.Resize((224, 224))(img)
resized.save("/content/colab_resized.png")

In [30]:
import numpy as np
from PIL import Image
import torchvision.transforms as transforms

# Load the exact same test image
image_path = f"{BASE}/test_ewaste.jpg"
img = Image.open(image_path).convert("RGB")

# Reproduce the model preprocessing
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

x = preprocess(img).unsqueeze(0)

# Save normalized tensor
np.save("/content/colab_input.npy", x.numpy())

# Save resized RGB image
resized = transforms.Resize((224, 224))(img)
resized.save("/content/colab_resized.png")

# Print statistics
print("Tensor shape:", x.shape)
print("Min:", x.min().item())
print("Max:", x.max().item())
print("Mean:", x.mean().item())
print("First 15 values:", x.flatten()[:15].tolist())

print("Files saved successfully!")

Tensor shape: torch.Size([1, 3, 224, 224])
Min: -1.878157377243042
Max: 2.4482791423797607
Mean: -0.02087908796966076
First 15 values: [0.7248052358627319, 0.7761794924736023, 0.7248052358627319, 0.6563062071800232, 0.7419300079345703, 0.9303022623062134, 1.1186745166778564, 1.18717360496521, 1.2727973461151123, 1.2214230298995972, 1.18717360496521, 1.18717360496521, 1.3584210872650146, 1.5467933416366577, 1.4954191446304321]
Files saved successfully!


In [31]:
from google.colab import files

files.download("/content/colab_input.npy")
files.download("/content/colab_resized.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [32]:
import numpy as np

arr = np.array(resized)

print("RGB shape:", arr.shape)
print("First 20 RGB pixels:")
print(arr.reshape(-1, 3)[:20].tolist())

RGB shape: (224, 224, 3)
First 20 RGB pixels:
[[166, 162, 145], [169, 163, 146], [166, 161, 143], [162, 155, 138], [167, 159, 142], [178, 170, 152], [189, 180, 162], [193, 184, 166], [198, 190, 172], [195, 185, 166], [193, 180, 159], [193, 179, 156], [203, 191, 171], [214, 204, 185], [211, 202, 181], [209, 200, 179], [213, 205, 185], [220, 213, 194], [225, 218, 199], [230, 222, 204]]


In [34]:
print("Original image:", img.size)
print("Original first 20 RGB pixels:")
print(np.array(img).reshape(-1, 3)[:20].tolist())

Original image: (1856, 4096)
Original first 20 RGB pixels:
[[238, 231, 213], [242, 235, 217], [244, 237, 219], [242, 235, 217], [239, 234, 215], [234, 229, 210], [227, 224, 205], [226, 223, 204], [230, 225, 206], [232, 227, 208], [224, 218, 202], [217, 211, 195], [212, 206, 192], [214, 208, 194], [218, 212, 198], [212, 206, 192], [230, 224, 208], [243, 237, 221], [241, 235, 219], [237, 231, 215]]


In [35]:
image_path = f"{BASE}/test_plastic_bottle.jpg"

In [38]:
import torch
from PIL import Image
import torchvision.transforms as transforms
import torch.nn.functional as F

# Load image
image_path = f"{BASE}/test_plastic_bottle.jpg"
img = Image.open(image_path).convert("RGB")

# Preprocessing (same as model training)
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Prepare input
x = preprocess(img).unsqueeze(0).to(device)

# Prediction
model.eval()

with torch.no_grad():
    output = model(x)
    probabilities = F.softmax(output, dim=1)[0]

# Class names
classes = [
    "ewaste",
    "glass",
    "hazardous",
    "metal",
    "organic",
    "paper_cardboard",
    "plastic_bottle_container",
    "plastic_film"
]

# Display results
predicted_idx = probabilities.argmax().item()

print("Predicted category:", classes[predicted_idx])
print(f"Confidence: {probabilities[predicted_idx].item() * 100:.2f}%")

print("\nAll class probabilities:")
for name, prob in zip(classes, probabilities):
    print(f"{name}: {prob.item() * 100:.2f}%")

Predicted category: plastic_film
Confidence: 58.74%

All class probabilities:
ewaste: 3.56%
glass: 7.50%
hazardous: 5.01%
metal: 18.21%
organic: 0.06%
paper_cardboard: 2.75%
plastic_bottle_container: 4.17%
plastic_film: 58.74%


In [37]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)